In [1]:
# === Cell 1: imports + settings ===
from __future__ import annotations

from pathlib import Path
import pickle

import numpy as np
import pandas as pd

# optioneel (pas later nodig)
from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

SyntaxError: incomplete input (2484863794.py, line 384)

In [ ]:
# === Cell 2: load pickle -> DataFrame ===
PICKLE_PATH = Path("results.pkl")   # <-- zet dit goed
DF_KEY = None                      # <-- als je pickle een dict is: bv "results_df"

def load_pickle(path: Path):
    with path.open("rb") as f:
        return pickle.load(f)

obj = load_pickle(PICKLE_PATH)

if isinstance(obj, pd.DataFrame):
    df = obj.copy()

elif isinstance(obj, dict):
    if DF_KEY is not None:
        if DF_KEY not in obj or not isinstance(obj[DF_KEY], pd.DataFrame):
            raise KeyError(f"DF_KEY={DF_KEY!r} not found (or not a DataFrame). Keys={list(obj.keys())}")
        df = obj[DF_KEY].copy()
    else:
        # probeer logische keys automatisch
        for k in ("results_df", "results", "df", "data"):
            if k in obj and isinstance(obj[k], pd.DataFrame):
                df = obj[k].copy()
                DF_KEY = k
                break
        else:
            raise KeyError(f"Pickle is dict but no DataFrame key found. Keys={list(obj.keys())}")

else:
    raise TypeError(f"Unsupported pickle content type: {type(obj)}")

print("Loaded:", PICKLE_PATH)
print("DF_KEY:", DF_KEY)
print("Shape:", df.shape)
df.head()

In [ ]:
# === Cell 3: basic sanity check + (optioneel) harmoniseren kolomnamen ===
# Verwacht (long format): sample_id, condition, + metric columns

# Als je andere namen gebruikt, map ze hier:
rename_map = {
    # "scenario_id": "sample_id",
    # "config": "condition",
}
if rename_map:
    df = df.rename(columns=rename_map)

required = ["sample_id", "condition"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Available columns:\n{df.columns.tolist()}")

df["sample_id"] = df["sample_id"].astype(str)
df["condition"] = df["condition"].astype(str)

print("Conditions:", sorted(df["condition"].unique())[:20], ("..." if df["condition"].nunique() > 20 else ""))
print("n_samples:", df["sample_id"].nunique(), "n_conditions:", df["condition"].nunique())
df[required].head()

In [ ]:
# === Cell 4: kies metrics + numeric coercion + clean subset ===
MEASURES = [
    # <-- vul met jouw kolommen
    "LLDA",
    "extrawork",
    "swaps",
]

present = [m for m in MEASURES if m in df.columns]
missing_m = [m for m in MEASURES if m not in df.columns]
if missing_m:
    print("WARNING: missing measures (skipped):", missing_m)
MEASURES = present
if not MEASURES:
    raise ValueError("No measures found. Set MEASURES to existing numeric columns.")

df_num = df.copy()
for m in MEASURES:
    df_num[m] = pd.to_numeric(df_num[m], errors="coerce")

df_clean = df_num.dropna(subset=["sample_id", "condition"] + MEASURES).copy()

print("Rows:", len(df), "->", len(df_clean), "after dropping NA in required/measures")
df_clean.head()

In [ ]:
# === Cell: Z-scores per sample (Vanwelsenaere-style) ===
# Assumes df_clean contains: sample_id, condition, + MEASURES (numeric)

def add_zscores_within_sample(df_long: pd.DataFrame, measures: list[str], sample_col: str = "sample_id") -> pd.DataFrame:
    out = df_long.copy()
    for m in measures:
        mu = out.groupby(sample_col)[m].transform("mean")
        sd = out.groupby(sample_col)[m].transform(lambda x: x.std(ddof=0))
        out[m + "_z"] = (out[m] - mu) / sd.replace(0, np.nan)
    return out

df_z = add_zscores_within_sample(df_clean, MEASURES)
ZCOLS = [m + "_z" for m in MEASURES]

# quick check
df_z[["sample_id", "condition"] + ZCOLS].head()

In [ ]:
# === Cell: Boxplots of Z-scores per condition (one plot per measure) ===
# Set a consistent order if you want:
# ORDER = ["A/0", "A/100", "A/200", "E/0", "E/100", "E/200"]
ORDER = None  # or list of condition labels

def boxplots_zscores(df_long: pd.DataFrame, measures: list[str], cond_col: str = "condition", order: list[str] | None = None):
    for m in measures:
        zc = m + "_z"
        d = df_long[[cond_col, zc]].dropna().copy()

        if order is None:
            conds = sorted(d[cond_col].unique())
        else:
            conds = [c for c in order if c in d[cond_col].unique()]

        groups = [d.loc[d[cond_col] == c, zc].to_numpy() for c in conds]

        plt.figure(figsize=(10, 4))
        plt.boxplot(groups, labels=conds, showmeans=True)
        plt.axhline(0.0, linestyle="--")
        plt.title(f"{m} (Z-scores within sample)")
        plt.ylabel("z-score [-]")
        plt.tight_layout()
        plt.show()

boxplots_zscores(df_z, MEASURES, order=ORDER)

In [ ]:
# === Cell (optional): Combined long-format table for reporting ===
# Median/mean of z-scores per condition, per measure
z_summary = (
    df_z.groupby("condition")[ZCOLS]
      .agg(["mean", "median", "std", "count"])
      .sort_index()
)
z_summary

In [ ]:
# === Cell: Helper long -> wide (required for Friedman/Wilcoxon) ===
# Assumes df_z exists and has columns: sample_id, condition, <measure>_z

def to_wide(df_long: pd.DataFrame, value_col: str,
            sample_col: str = "sample_id",
            cond_col: str = "condition") -> pd.DataFrame:
    wide = df_long.pivot_table(index=sample_col, columns=cond_col, values=value_col, aggfunc="mean")
    # complete-case only (Friedman requires paired observations across all included conditions)
    return wide.dropna(axis=0, how="any")

In [ ]:
# === Cell: Friedman ANOVA (SciPy friedmanchisquare) + Kendall's W ===
# Runs per measure on the Z-scored columns.

def friedman_test_for_measure(df_long: pd.DataFrame, measure: str,
                              order: list[str] | None = None) -> dict:
    zc = measure + "_z"
    wide = to_wide(df_long, zc)

    if order is not None:
        cols = [c for c in order if c in wide.columns]
        wide = wide[cols].dropna(axis=0, how="any")

    # Need >=3 conditions and >=3 samples for a meaningful Friedman test
    if wide.shape[1] < 3 or wide.shape[0] < 3:
        return {
            "measure": measure,
            "chi2": np.nan,
            "p": np.nan,
            "n_samples": int(wide.shape[0]),
            "k_conditions": int(wide.shape[1]),
            "kendall_W": np.nan,
            "conditions": list(wide.columns),
        }

    arrays = [wide[c].to_numpy() for c in wide.columns]
    chi2, p = stats.friedmanchisquare(*arrays)

    n = wide.shape[0]
    k = wide.shape[1]
    kendall_W = float(chi2) / (n * (k - 1))  # effect size

    return {
        "measure": measure,
        "chi2": float(chi2),
        "p": float(p),
        "n_samples": int(n),
        "k_conditions": int(k),
        "kendall_W": float(kendall_W),
        "conditions": list(wide.columns),
    }

def friedman_all_measures(df_long: pd.DataFrame, measures: list[str],
                          order: list[str] | None = None) -> pd.DataFrame:
    rows = [friedman_test_for_measure(df_long, m, order=order) for m in measures]
    out = pd.DataFrame(rows).sort_values("p")
    return out

# Optional: enforce a consistent condition order (recommended)
# ORDER = ["A/0","A/100","A/200","E/0","E/100","E/200"]
ORDER = None

friedman_results = friedman_all_measures(df_z, MEASURES, order=ORDER)
friedman_results

In [ ]:
# === Cell (optional): flag significant results at alpha ===
alpha = 0.05
friedman_results.assign(significant=lambda d: d["p"] < alpha)

In [ ]:
# === Cell: Planned post-hoc Wilcoxon signed-rank tests (+ correction) ===
# Assumes:
# - df_z exists
# - MEASURES exists
# - each measure has a z-column: f"{measure}_z"
# - columns: sample_id, condition

# EDIT: planned comparisons (as in the report / your design)
PLANNED_PAIRS = [
    ("A/100", "A/0"),
    ("A/100", "A/200"),
    ("E/100", "E/0"),
    ("E/100", "E/200"),
    ("A/100", "E/100"),
]

CORRECTION = "bonferroni"  # "bonferroni" or "holm"
ALTERNATIVE = "two-sided"  # "two-sided", "greater", "less"

def wilcoxon_posthoc(df_long: pd.DataFrame,
                     measures: list[str],
                     pairs: list[tuple[str, str]],
                     correction: str = "bonferroni",
                     alternative: str = "two-sided",
                     order: list[str] | None = None) -> pd.DataFrame:
    rows = []

    for m in measures:
        zc = m + "_z"
        wide = to_wide(df_long, zc)

        # optional: enforce order subset
        if order is not None:
            cols = [c for c in order if c in wide.columns]
            wide = wide[cols].dropna(axis=0, how="any")

        for a, b in pairs:
            if a not in wide.columns or b not in wide.columns:
                continue

            x = wide[a].to_numpy()
            y = wide[b].to_numpy()

            # wilcoxon needs paired observations
            if len(x) < 3:
                continue

            stat, p = stats.wilcoxon(
                x, y,
                zero_method="wilcox",    # drop zero diffs
                alternative=alternative,
                mode="auto"
            )

            d = x - y
            rows.append({
                "measure": m,
                "pair": f"{a} vs {b}",
                "n": int(len(d)),
                "stat": float(stat),
                "p_raw": float(p),
                "mean_diff_z": float(np.mean(d)),
                "median_diff_z": float(np.median(d)),
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    method = "bonferroni" if correction.lower() == "bonferroni" else "holm"
    reject, p_adj, _, _ = multipletests(out["p_raw"].to_numpy(), method=method)

    out["p_adj"] = p_adj
    out["reject"] = reject
    return out.sort_values(["measure", "p_adj", "p_raw"])

posthoc = wilcoxon_posthoc(
    df_z, MEASURES, PLANNED_PAIRS,
    correction=CORRECTION,
    alternative=ALTERNATIVE,
    order=ORDER,  # can be None
)

posthoc

In [ ]:
# === Cell: Pretty summary per measure (which pairs significant) ===
alpha = 0.05
sig = posthoc[posthoc["p_adj"] < alpha].copy()

summary_pairs = (
    sig.groupby("measure")["pair"]
       .apply(list)
       .rename("significant_pairs")
)

summary = friedman_results.merge(summary_pairs, on="measure", how="left")
summary["significant_pairs"] = summary["significant_pairs"].apply(lambda x: x if isinstance(x, list) else [])
summary

In [ ]:
# === Cell (optional): export tables ===
OUTDIR = Path("stat_results")
OUTDIR.mkdir(exist_ok=True)

friedman_results.to_csv(OUTDIR / "friedman_main_effects.csv", index=False)
posthoc.to_csv(OUTDIR / f"wilcoxon_posthoc_{CORRECTION}.csv", index=False)
summary.to_csv(OUTDIR / f"summary_{CORRECTION}.csv", index=False)

print("Wrote:", OUTDIR.resolve())